# Scaling Laws

This notebook contains a set of experiments to discover scaling laws on a character-tokenized dataset and a small language model based on the encoder-only transformer architecture.

References:

https://arxiv.org/abs/2001.08361

https://arxiv.org/pdf/2203.15556

https://github.com/BenAgro314/MinChilla?tab=readme-ov-file

https://unsloth.ai/docs

https://github.com/mlabonne/llm-course

## Utils

### Model definition

In [1]:
import torch.nn as nn
import torch
import math
from dataclasses import dataclass


torch.manual_seed(1233)


@dataclass
class ModelConfig:
    vocab_size: int = 64
    n_embedding: int = 1024
    n_layers: int = 2
    n_heads: int = 4
    context_window: int = 128


class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding from "Attention is All You Need".
    """
    def __init__(self, n_embedding, context_window):
        super().__init__()
        self.pe = torch.zeros(context_window, n_embedding)
        position = torch.arange(0, context_window, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, n_embedding, 2).float() * (-math.log(10000.0) / n_embedding)
        )
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = self.pe.unsqueeze(0)  # shape: (1, context_window, n_embedding)

    def forward(self, x):
        """
        x: shape [batch_size, seq_len, n_embedding]
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :].to(x.device)
    

class SmallTransformer(nn.Module):
    """
    Small Transformer Based Model
    """

    def __init__(self, model_config: ModelConfig):

        super().__init__()
        
        # model hyper-parameters
        self.vocab_size = model_config.vocab_size
        self.n_embedding = model_config.n_embedding
        self.n_layers = model_config.n_layers
        self.n_heads = model_config.n_heads
        self.context_window = model_config.context_window

        # embeddings
        self.token_embedding = nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=self.n_embedding)
        self.positional_embedding = PositionalEncoding(context_window=self.context_window, n_embedding=self.n_embedding)

        # N transforme layers
        dim_feedforward = self.n_embedding * 2
        encoder_layer = nn.TransformerEncoderLayer(nhead=self.n_heads, d_model=self.n_embedding, dim_feedforward=dim_feedforward, batch_first=True)
        layer_norm = nn.LayerNorm(self.n_embedding)
        self.transformer_blocks = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=self.n_layers, norm=layer_norm)

        # Linear Head
        self.lm_head = nn.Linear(self.n_embedding, self.vocab_size)

    def forward(self, x):
        """
        x: shape [batch_size, seq_len, n_embedding]
        """
        out = self.token_embedding(x)
        out = self.positional_embedding(out)
        out = self.transformer_blocks(out)
        out = self.lm_head(out)
        return out

### Data Processing

In [2]:
from torch.utils.data import Dataset, DataLoader


class CharacterDataset(Dataset):
    """
    Creates a character-level dataset by slicing a single long token tensor.
    Returns (x, y) pairs where x and y are arrays of integer IDs for chars.
    """
    def __init__(self, tokens, seq_len):
        super().__init__()
        self.tokens = tokens
        self.seq_len = seq_len

    def __len__(self):
        # We'll generate slices up to len(tokens)-seq_len
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        x = self.tokens[idx : idx + self.seq_len]
        y = self.tokens[idx + 1 : idx + self.seq_len + 1]
        return x, y


class CharacterTokenizer:

    def __init__(self):
        self.char2idx = {}
        self.idx2char = {}
        self._vocab_size = None

    def train(self, train_data):
        """
        Builds char2idx and idx2char mappings given a list of strings.
        """
        # Collect all characters
        unique_chars = set("".join([item['text'] for item in train_data]))
        # Ensure a padding character exists
        unique_chars.add(' ')  # Adding space as padding if not present
        # Sort for reproducibility
        unique_chars = sorted(list(unique_chars))
        self.char2idx = {ch: i for i, ch in enumerate(unique_chars)}
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}

        self._vocab_size = len(self.char2idx)

        print(f"Built vocabulary with {self._vocab_size} characters.")
    
    @property
    def vocab_size(self):
        return self._vocab_size
    
    def encode(self, text):
        tokens = torch.tensor([self.char2idx.get(ch, self.char2idx[' ']) for ch in text], dtype=torch.long)
        return tokens
    
    def decode(self, tokens):
        text = "".join([self.idx2char.get(tok, self.idx2char[self.char2idx[' ']]) for tok in tokens])
        return text

### Training & Eval utils

In [ ]:

from torch.optim import AdamW
import torch.nn.functional as F


def criterion(y, targets):
    """
    Compute Cross Entropy
    """
    loss = F.cross_entropy(y, targets)
    return loss


from tqdm import tqdm


def train_one_epoch(optimizer, model, train_data_loader: DataLoader, epoch: int, log_file_name: str):

    # activate train mode
    model.train()

    progress_bar = tqdm(enumerate(train_data_loader), total=len(train_data_loader), desc=f"Epoch {epoch+1} [Train]")

    for batch_idx, (x, y) in progress_bar:

        # remove old gradients
        optimizer.zero_grad()

        # compute loss
        logits = model(x)
        train_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        # compute gradients
        train_loss.backward()

        # back-propagate
        optimizer.step()

        with open(log_file_name, "a") as f:
            f.write(f"Iteration: {batch_idx} | Training Loss: {train_loss:.2f}\n")


@torch.no_grad()
def evaluate(model, validation_data_loader, epoch: int, log_file_name: str):

    # activate eval mode
    model.eval()

    progress_bar = tqdm(enumerate(validation_data_loader), total=len(validation_data_loader), desc=f"Epoch {epoch+1} [Validation]")

    for batch_idx, (x, y) in progress_bar:

        # compute loss
        logits = model(x)
        val_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        with open(log_file_name, "a") as f:
            f.write(f"Iteration: {batch_idx} | Validation Loss: {val_loss:.2f}\n")


## Main Code

### Load, Tokenize

In [7]:
from datasets import load_dataset

# -----------------------
train_data = load_dataset("roneneldan/TinyStories", split="train[:5]")
validation_data = load_dataset("roneneldan/TinyStories", split="validation[:2]")

# -----------------------
tokenizer = CharacterTokenizer()
tokenizer.train(train_data)

train_tokens = torch.concat([tokenizer.encode(item['text']) for item in train_data])
validation_tokens = torch.concat([tokenizer.encode(item['text']) for item in validation_data])

Built vocabulary with 49 characters.


### Model

In [ ]:
# -----------------------
model_config = ModelConfig(vocab_size=tokenizer.vocab_size, n_embedding=256)
print(model_config)
model = SmallTransformer(model_config)
print(f"Number of model parameters: {sum([l.numel() for l in model.parameters()])/1e6:.2f}")


### Training Loop

In [ ]:
# -----------------------
@dataclass
class TrainingConfig:
    max_epochs: int = 1
    lr: float = 3e-4
    batch_size: int = 128

training_config = TrainingConfig()
optimizer = AdamW(model.parameters(), lr=training_config.lr)

# -----------------------
train_dataset = CharacterDataset(train_tokens, seq_len=model_config.context_window)
validation_dataset = CharacterDataset(validation_tokens, seq_len=model_config.context_window)
train_data_loader = DataLoader(
    train_dataset,
    batch_size=training_config.batch_size,
)
validation_data_loader = DataLoader(
    validation_dataset,
    batch_size=training_config.batch_size
)

# -----------------------
log_file_name = "scaling_laws.txt"
log_file = open(log_file_name, "w")
metrics = []

for epoch in range(training_config.max_epochs):

    train_one_epoch(
        optimizer=optimizer,
        model=model,
        train_data_loader=train_data_loader,
        epoch=epoch,
        log_file_name=log_file_name,
    )

    evaluate(
        model=model,
        validation_data_loader=validation_data_loader,
        epoch=epoch,
        log_file_name=log_file_name
    )



In [ ]:
# From karpathy/nanoGPT repository (scaling_laws.ipynb)
# {Link: scaling_laws.ipynb https://github.com/karpathy/nanoGPT/blob/master/scaling_laws.ipynb}

import numpy as np
from scipy.optimize import curve_fit
# from nanoGPT.utils import chinchilla_params # Helper function for params

# 1. Define the scaling law function
def power_law(n, d, alpha_n, alpha_d, c_n, c_d, l_inf):
    return c_n * (n ** -alpha_n) + c_d * (d ** -alpha_d) + l_inf

# 2. Sample data (replace with your actual training runs)
# Example: (params, tokens, loss)
training_results = [
    (100000, 1000000, 2.5),
    (500000, 5000000, 1.8),
    (2000000, 20000000, 1.2),
]
N_vals = np.array([r[0] for r in training_results])
D_vals = np.array([r[1] for r in training_results])
L_vals = np.array([r[2] for r in training_results])

# 3. Fit the curve (using initial guesses for parameters)
params, covariance = curve_fit(
    lambda n, d, alpha_n, alpha_d, c_n, c_d, l_inf: power_law(n, d, alpha_n, alpha_d, c_n, c_d, l_inf),
    (N_vals, D_vals), L_vals, p0=[0.1, 0.1, 10, 10, 0.1]
)

print(f"Fitted Params: {params}")

# 4. Use fitted params to predict for new (N, D) or find optimal allocation
# ... (Further code to calculate optimal N, D for a budget C)


ModuleNotFoundError: No module named 'scipy'